Implementing "Decision Tree":
 We are analyzing the data of the world-cups until 2018 and predict the 2022 outcome cup

1. Preparing the data by clarifying the test data and preparing the data which we want to predict (2022 Cup)


In [1]:
import pandas as pd

df = pd.read_csv("../test_analysis/ml_df_test_bab.csv")

#Using WM untlil 2018 for training, trying to predict Cup-2022
df_train = df[df["year"] <= 2018]
prediction = df[df['year'] == 2022]

#We want to predict which team will win the match (Home or Away)
y_train = df_train["result_target"]

#Chosing the relevant columns for the prediction (,without spoiler)
# I let gemini choose the relevant columns, because I want to avoid data leaks, and it is hard to chose which columns are "spoiler" columns
relevant_columns =[
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    
    'home_elo_before', 'away_elo_before',
    'home_total_win_rate_before', 'away_total_win_rate_before',
    'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before',
    
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    

    'home_tournaments_played_before', 'away_tournaments_played_before',
    'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    'home_is_defending_champion', 'away_is_defending_champion'
]

x_train = df_train[relevant_columns]

#Machine cant work with strings, so we are dropping all columns with words:
#(for example: "match name" is being dropped, instead we use "match_id")
x_train = x_train.select_dtypes(exclude=['object'])

x_train.head()


#Testing the prediction on the 2022 cup,
#Preparing the 2022 data for the prediction
y_test = prediction["result_target"]
x_test = prediction[relevant_columns]
x_test = x_test.select_dtypes(exclude=['object'])

2. Implementing our prepared data and training our data with the cup of 2018

In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

#Clarifying the max depth, so that the model doesnt analyze in too much detail
tree_model = DecisionTreeClassifier(max_depth = 5,random_state=42)
tree_model.fit(x_train, y_train)

#Now the model is trained with the data until 2018,

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

3. Predicting the 2022 Cup with the trained model

In [3]:
#Now we want to test the model on the 2022 cup, 
y_pred = tree_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

#average = "weighted" is used to calculate with focus on the more common type of result
#(it is more important to get the more common result right, than the less common one) 
# -> Alternative: for example"macro": all results are equally important: 33,33%
f1 = f1_score(y_test, y_pred, average='weighted')


print(f"____Prediction for 2022 World Cup____")
print(f"Accuracy: {accuracy:.2%}") #rounded by 2 decimals, shown in percentage
print(f"F1-Score: {f1:.2%}") 

final_match_prediction = y_pred[-1]
print(f"Das Modell sagt für das Finale 2022: {final_match_prediction}")

print(x_train.columns.tolist())



____Prediction for 2022 World Cup____
Accuracy: 43.75%
F1-Score: 44.12%
Das Modell sagt für das Finale 2022: HomeWin
['elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 'goals_per_match_diff', 'conceded_per_match_diff', 'home_elo_before', 'away_elo_before', 'home_total_win_rate_before', 'away_total_win_rate_before', 'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before', 'home_last5_win_rate', 'away_last5_win_rate', 'home_last5_goal_diff', 'away_last5_goal_diff', 'home_tournaments_played_before', 'away_tournaments_played_before', 'home_has_won_world_cup_before', 'away_has_won_world_cup_before', 'home_is_defending_champion', 'away_is_defending_champion']


In [4]:
#Helped us looking for data leaks, it shows us which columns the model uses for prediction

helpful_columns = pd.Series(tree_model.feature_importances_, index=x_train.columns)

print("\n Most used columns for the prediction: \n")
print(helpful_columns.sort_values(ascending=False).head(5))

print(df.columns.tolist())


#COPY PASTE FROM GEMINI FOR NOW; just to check if the predction is correct
final_spiel = prediction.iloc[-1]
heim_team = final_spiel['home_team']
auswaerts_team = final_spiel['away_team']

print(f"\n--- DAS FINALE 2022 ---")
print(f"Auf dem Platz stehen: {heim_team} (Home) vs. {auswaerts_team} (Away)")

# 3. Wir übersetzen den Tipp des Modells in den Teamnamen
if final_match_prediction == 'HomeWin':
    print(f"🏆 Das Modell tippt auf Weltmeister: {heim_team}!")
elif final_match_prediction == 'AwayWin':
    print(f"🏆 Das Modell tippt auf Weltmeister: {auswaerts_team}!")
else:
    print(f"🤝 Das Modell tippt auf ein Unentschieden nach 90 Minuten!")


 Most used columns for the prediction: 

away_total_win_rate_before         0.349708
away_goal_diff_per_match_before    0.168354
away_tournaments_played_before     0.129591
away_elo_before                    0.076512
win_rate_diff                      0.063396
dtype: float64
['year', 'date', 'tournament_id', 'tournament_name', 'match_name', 'stage', 'home_team', 'away_team', 'home_team_code', 'away_team_code', 'home_goals', 'away_goals', 'score', 'result', 'home_team_win', 'away_team_win', 'draw', 'extra_time', 'penalty_shootout', 'score_penalties', 'stadium', 'stadium_id', 'stadium_name', 'city', 'host_country', 'host_team', 'attendance', 'match_time', 'referee', 'notes', 'match_id', 'total_teams', 'matches_played', 'goals_scored_tournament', 'avg_goals_per_game', 'year_normalized', 'tournament_size_category', 'home_total_matches_before', 'home_total_wins_before', 'home_total_draws_before', 'home_total_losses_before', 'home_total_goals_for_before', 'home_total_goals_against_before', 

Ways of Improving the model: 

In [5]:
print("\n Classification Report: \n" , classification_report(y_test, y_pred))



 Classification Report: 
               precision    recall  f1-score   support

     AwayWin       0.45      0.45      0.45        20
        Draw       0.12      0.13      0.13        15
     HomeWin       0.61      0.59      0.60        29

    accuracy                           0.44        64
   macro avg       0.39      0.39      0.39        64
weighted avg       0.45      0.44      0.44        64



1. Improvement

If we look at the report above, we see that the models precision for a draw is only at 13%, what is really bad. 
It is harder for the model to predict a draw than a Win/Lose, so it rather guessing which team wins than guessing for a draw, so the probability of guessing right stays higher.
So what happens if we "punish"/"reward" wrong/right guesses on "Draw" more, so that the model rather wants to risk guessing it rightfully than having it wrong ? :


In [6]:
tree_model = DecisionTreeClassifier(
    max_depth=5,               
    class_weight='balanced',     #Rare results (like "Draw") are rewarded/punished more
    random_state=42
)
# Training data until Worldcup 2018
tree_model.fit(x_train, y_train)

# Prediction of 2022 Worldcup
y_pred = tree_model.predict(x_test)

print("\n Classification Report with balanced class weights: \n" , classification_report(y_test, y_pred))   


 Classification Report with balanced class weights: 
               precision    recall  f1-score   support

     AwayWin       0.58      0.55      0.56        20
        Draw       0.38      0.20      0.26        15
     HomeWin       0.62      0.79      0.70        29

    accuracy                           0.58        64
   macro avg       0.53      0.51      0.51        64
weighted avg       0.55      0.58      0.55        64






2. Improvement: 
Checking which parameters are the best for the model, combining it with the 1. Improvement - "class_weight= balanced"

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier


parameter = {
    'max_depth': [3, 4, 5, 6, 7],         #Depth of the tree, searching the ideal depth because too much/less is bad for the model (overfitting/not enough detail)    
    'min_samples_split': [2, 5, 10, 20],  #Minimum number of teams that must be in a node to be allowed tosplit it    

    'min_samples_leaf': [1, 2, 5, 10],    #Minimum number of team allowed in leafnode 
                                          #-> it prevents the model from overfitting, because it doesnt allow the model to analyze too much detail, 
                                          #   and it forces the model to generalize more
    'class_weight': ['balanced']   #"balanced": Rare results (like "Draw") rewarded/punish more
}
#Algorithm tests 5*4*4*2 = 160 different combinations of parameters, to find the best one for our model

new_testing_tree = DecisionTreeClassifier(random_state=42)

search_algorithm = GridSearchCV(
    estimator=new_testing_tree, #Which model to test?
    param_grid=parameter, #Testing with which parameters?
    scoring='f1_macro',   # "scoring": Choses the criteria, f1_macro: all results are equally important
    cv = 5,               # "Cross validation": the data is split into 5 parts, so the model is trained 5 times, each time with a different part as test data 
    n_jobs=-1             #Use all CPU cores for parallel processing
)

search_algorithm.fit(x_train, y_train)  #Includes many different trained trees
print(search_algorithm.best_params_)    #Shows the best parameters for the model
best_tree_model = search_algorithm.best_estimator_ #Model with the best parameters, 


{'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}


Comparing the old and the new model: 

In [8]:
y_pred_old = tree_model.predict(x_test)
y_pred_new = best_tree_model.predict(x_test)

#Classification report for the old model:
print("\n Classification Report for the old model: \n" , classification_report(y_test, y_pred_old))
#Classification report for the new model:
print("\n Classification Report for the new model: \n" , classification_report(y_test, y_pred_new))


 Classification Report for the old model: 
               precision    recall  f1-score   support

     AwayWin       0.58      0.55      0.56        20
        Draw       0.38      0.20      0.26        15
     HomeWin       0.62      0.79      0.70        29

    accuracy                           0.58        64
   macro avg       0.53      0.51      0.51        64
weighted avg       0.55      0.58      0.55        64


 Classification Report for the new model: 
               precision    recall  f1-score   support

     AwayWin       0.54      0.65      0.59        20
        Draw       0.50      0.20      0.29        15
     HomeWin       0.65      0.76      0.70        29

    accuracy                           0.59        64
   macro avg       0.56      0.54      0.53        64
weighted avg       0.58      0.59      0.57        64



Improving our prediction by starting with RandomForest:

In [9]:
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import classification_report

# Empty Forest:
forest_test = RandomForestClassifier(
    n_estimators=100,   #Number of trees in the forest, more trees -> better results, but it takes longer
    #Paramteters for every single tree in the forest:
    max_depth=5,
    class_weight='balanced',
    random_state=42
)

#Training the 100 trees in the forest 
print("Planting the 100 trees:")
forest_test.fit(x_train, y_train) #Fit gives every tree random columns to learn from 
                                  #-> forces trees to be different, to be "creative" and not lazy

#Using the trained forest to predict the 2022 cup
y_pred_forest = forest_test.predict(x_test)
print("Prediction completed!")

print("\n Classification Report for the Random Forest model: \n" , classification_report(y_test, y_pred_forest))


Planting the 100 trees:
Prediction completed!

 Classification Report for the Random Forest model: 
               precision    recall  f1-score   support

     AwayWin       0.48      0.65      0.55        20
        Draw       0.24      0.27      0.25        15
     HomeWin       0.70      0.48      0.57        29

    accuracy                           0.48        64
   macro avg       0.47      0.47      0.46        64
weighted avg       0.52      0.48      0.49        64



If we compare our best single tree model with the Forest model, we will see that the tree had way better results than the forest. 
Comparison below:

In [11]:

print("\n Classification Report for the improved Tree model: \n" , classification_report(y_test, y_pred_new))

print("\n Classification Report for the Random Forest model: \n" , classification_report(y_test, y_pred_forest))



 Classification Report for the improved Tree model: 
               precision    recall  f1-score   support

     AwayWin       0.54      0.65      0.59        20
        Draw       0.50      0.20      0.29        15
     HomeWin       0.65      0.76      0.70        29

    accuracy                           0.59        64
   macro avg       0.56      0.54      0.53        64
weighted avg       0.58      0.59      0.57        64


 Classification Report for the Random Forest model: 
               precision    recall  f1-score   support

     AwayWin       0.48      0.65      0.55        20
        Draw       0.24      0.27      0.25        15
     HomeWin       0.70      0.48      0.57        29

    accuracy                           0.48        64
   macro avg       0.47      0.47      0.46        64
weighted avg       0.52      0.48      0.49        64



In [12]:
import pandas as pd

df = pd.read_csv("../test_analysis/ml_df_test_bab.csv")


#Using WM untlil 2018 for training, trying to predict Cup-2022
df_train = df[df["year"] <= 2018]
prediction = df[df['year'] == 2022]

#We want to predict which team will win the match (Home or Away)
y_train = df_train["result_target"]

#Chosing the relevant columns for the prediction (,without spoiler)
# I let gemini choose the relevant columns, because I want to avoid data leaks, and it is hard to chose which columns are "spoiler" columns
relevant_columns =[
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    
    'home_elo_before', 'away_elo_before',
    'home_total_win_rate_before', 'away_total_win_rate_before',
    'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before',
    
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    

    'home_tournaments_played_before', 'away_tournaments_played_before',
    'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    'home_is_defending_champion', 'away_is_defending_champion'
]

x_train = df_train[relevant_columns]
x_train = x_train.select_dtypes(exclude=['object'])

y_test = prediction["result_target"]
x_test = prediction[relevant_columns]
x_test = x_test.select_dtypes(exclude=['object'])

tree_model = DecisionTreeClassifier(
    max_depth=3,               
    class_weight='balanced', 
    min_samples_leaf=1,    #Rare results (like "Draw") are rewarded/punished more
    min_samples_split=2,
    random_state=42
)

tree_model.fit(x_train, y_train)
y_pred = tree_model.predict(x_test)

print("\n Classification Report for the old model: \n" , classification_report(y_test, y_pred))





 Classification Report for the old model: 
               precision    recall  f1-score   support

     AwayWin       0.54      0.65      0.59        20
        Draw       0.50      0.20      0.29        15
     HomeWin       0.65      0.76      0.70        29

    accuracy                           0.59        64
   macro avg       0.56      0.54      0.53        64
weighted avg       0.58      0.59      0.57        64

